# Flickr30k T3→S3 CPU Proxy Benchmark

Đo thời gian tiền xử lý, encoder FP32, decoder INT8 Beam-3 và end-to-end trên
50 ảnh Validation với ONNX Runtime CPU, lần lượt dùng 1 và tối đa 4 thread.

Đây chỉ là **proxy x86 trên Kaggle**, không phải kết quả Android. Không được
dùng số liệu này để tuyên bố latency, RAM, nhiệt độ hoặc năng lượng trên điện
thoại.

## Kaggle input bắt buộc — đúng 4 input

1. Output `F30K_T3S3_Deployment_Lock`.
2. Output `F30K_T3S3_INT8_Validation`.
3. Dataset `Flickr Image dataset`.
4. Dataset `ONNX_Runtime_Python313_Offline`.

Runtime: CPU / Accelerator None / Internet Off, dự kiến 3–8 phút.
Held-out Test tiếp tục niêm phong.

## Cell 1 — Package authorization and runtime

In [1]:
import hashlib
import importlib.util
import json
import math
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

EXPERIMENT_ID = "F30K_T3S3_CPU_Proxy"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

LOCK_RESULT_PATH = unique_named(
    "F30K_T3S3_Deployment_Lock_RESULT.json"
)
CONFIG_PATH = unique_named("F30K_T3S3_MOBILE_CONFIG.json")
CAPTIONS_PATH = unique_named(
    "F30K_T3S3_INT8_Validation_CAPTIONS.csv"
)
ENCODER_PATH = unique_named("f30k_t3s3_seed42_encoder_fp32.onnx")
DECODER_PATH = unique_named("f30k_t3s3_seed42_decoder_dynamic_int8.onnx")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")

LOCK = read_json(LOCK_RESULT_PATH)
CONFIG = read_json(CONFIG_PATH)
VOCAB = read_json(VOCAB_PATH)
assert LOCK["status"] == "locked_for_conditional_mobile_benchmark"
assert LOCK["decision"]["mobile_benchmark"] == "authorized"
assert LOCK["decision"]["unconditional_production_release"] == "not authorized"
assert LOCK["decision"]["strict_verbatim_stability"] == "failed"
assert LOCK["test_set_status"] == "sealed"
assert LOCK["test_set_used"] is False

artifact_by_name = {
    Path(record["relative_path"]).name: record
    for record in LOCK["package_artifacts"]
}
for path in [ENCODER_PATH, DECODER_PATH, VOCAB_PATH, CONFIG_PATH]:
    record = artifact_by_name[path.name]
    assert path.stat().st_size == record["size_bytes"]
    assert sha256(path) == record["sha256"]
assert CONFIG["vocabulary"]["size"] == len(VOCAB["tokens"]) == 7456

if importlib.util.find_spec("onnxruntime") is None:
    wheels = sorted(INPUT_ROOT.rglob("onnxruntime*.whl"))
    assert wheels, "Hãy gắn dataset ONNX_Runtime_Python313_Offline."
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
    )
import onnxruntime as ort
try:
    import psutil
except ImportError:
    psutil = None

print("Authorization          : PASSED")
print("Benchmark role         : x86 CPU PROXY, NOT ANDROID")
print("Package status         :", LOCK["status"])
print("ONNX Runtime           :", ort.__version__)
print("Host CPU count         :", os.cpu_count())
print("Held-out Test          : SEALED / NOT READ")

Authorization          : PASSED
Benchmark role         : x86 CPU PROXY, NOT ANDROID
Package status         : locked_for_conditional_mobile_benchmark
ONNX Runtime           : 1.28.0
Host CPU count         : 4
Held-out Test          : SEALED / NOT READ


## Cell 2 — Deterministic Validation sample and inference

In [2]:
captions = pd.read_csv(CAPTIONS_PATH)
assert len(captions) == 1014 and captions["image_id"].nunique() == 1014
sorted_ids = sorted(captions["image_id"].astype(int).tolist())
sample_positions = np.linspace(0, len(sorted_ids) - 1, 50, dtype=np.int64)
SAMPLE_IDS = [sorted_ids[int(position)] for position in sample_positions]
assert len(SAMPLE_IDS) == len(set(SAMPLE_IDS)) == 50

needed_names = {f"{image_id}.jpg" for image_id in SAMPLE_IDS}
IMAGE_BY_NAME = {}
for path in INPUT_ROOT.rglob("*.jpg"):
    if path.name not in needed_names:
        continue
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
    else:
        assert old.stat().st_size == path.stat().st_size
        IMAGE_BY_NAME[path.name] = min(
            [old, path],
            key=lambda item: (len(item.parts), len(str(item)), str(item)),
        )
assert set(IMAGE_BY_NAME) == needed_names

MEAN = np.asarray(CONFIG["preprocessing"]["mean"],
                  dtype=np.float32).reshape(1, 1, 3)
STD = np.asarray(CONFIG["preprocessing"]["std"],
                 dtype=np.float32).reshape(1, 1, 3)
TOKENS = VOCAB["tokens"]
SPECIAL = {0, 1, 2, 3}

def resize_shorter_side(image, target=256):
    width, height = image.size
    if width <= height:
        shape = (target, int(target * height / width))
    else:
        shape = (int(target * width / height), target)
    return image.resize(shape, resample=Image.Resampling.BICUBIC)

def preprocess(path):
    with Image.open(path) as opened:
        image = resize_shorter_side(opened.convert("RGB"), 256)
    width, height = image.size
    left = int(round((width - 224) / 2.0))
    top = int(round((height - 224) / 2.0))
    image = image.crop((left, top, left + 224, top + 224))
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - MEAN) / STD
    return np.transpose(array, (2, 0, 1))[None].astype(np.float32)

def stable_log_softmax(values):
    values = np.asarray(values, dtype=np.float64)
    maximum = float(values.max())
    shifted = values - maximum
    return shifted - math.log(float(np.exp(shifted).sum()))

def normalized_score(beam):
    return beam[1] / (max(1, len(beam[0]) - 1) ** 0.7)

def beam_decode(session, memory):
    beams = [([1], 0.0, False)]
    calls = 0
    for _ in range(30):
        active = [beam for beam in beams if not beam[2]]
        if not active:
            break
        candidates = [beam for beam in beams if beam[2]]
        for token_ids, score, _ in active:
            token_buffer = np.zeros((1, 31), dtype=np.int64)
            attention_mask = np.zeros((1, 31), dtype=np.bool_)
            length = len(token_ids)
            token_buffer[0, :length] = token_ids
            attention_mask[0, :length] = True
            logits = session.run(
                ["logits"],
                {"token_ids": token_buffer, "memory": memory,
                 "attention_mask": attention_mask},
            )[0][0, length - 1].astype(np.float64)
            calls += 1
            logits[0] = logits[1] = -np.inf
            logp = stable_log_softmax(logits)
            top = np.argpartition(logp, -3)[-3:]
            top = sorted(
                (int(index) for index in top),
                key=lambda index: (-float(logp[index]), index),
            )
            for index in top:
                candidates.append((
                    token_ids + [index],
                    score + float(logp[index]),
                    index == 2,
                ))
        beams = sorted(
            candidates,
            key=lambda beam: (-normalized_score(beam), tuple(beam[0])),
        )[:3]
    return min(
        beams,
        key=lambda beam: (-normalized_score(beam), tuple(beam[0])),
    )[0], calls

def decode(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id not in SPECIAL:
            words.append(TOKENS[token_id])
    text = " ".join(words)
    for punctuation in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + punctuation, punctuation)
    return text.strip()

print("Benchmark images       :", len(SAMPLE_IDS))
print("Sampling               : 50 evenly spaced Validation image IDs")
print("Held-out Test          : SEALED / NOT READ")

Benchmark images       : 50
Sampling               : 50 evenly spaced Validation image IDs
Held-out Test          : SEALED / NOT READ


## Cell 3 — Warm-up and latency benchmark

In [3]:
def percentile_summary(values):
    array = np.asarray(values, dtype=np.float64)
    return {
        "count": int(len(array)),
        "mean_ms": float(array.mean()),
        "std_ms": float(array.std(ddof=1)),
        "p50_ms": float(np.percentile(array, 50)),
        "p90_ms": float(np.percentile(array, 90)),
        "p95_ms": float(np.percentile(array, 95)),
        "p99_ms": float(np.percentile(array, 99)),
        "min_ms": float(array.min()),
        "max_ms": float(array.max()),
    }

thread_values = sorted(set([1, min(4, max(1, os.cpu_count() or 1))]))
all_rows = []
summaries = {}
process = psutil.Process(os.getpid()) if psutil is not None else None

for thread_count in thread_values:
    options = ort.SessionOptions()
    options.intra_op_num_threads = thread_count
    options.inter_op_num_threads = 1
    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    started = time.perf_counter()
    encoder = ort.InferenceSession(
        str(ENCODER_PATH), sess_options=options,
        providers=["CPUExecutionProvider"],
    )
    decoder = ort.InferenceSession(
        str(DECODER_PATH), sess_options=options,
        providers=["CPUExecutionProvider"],
    )
    load_ms = (time.perf_counter() - started) * 1000.0

    # Warm-up không đưa vào thống kê.
    for image_id in SAMPLE_IDS[:3]:
        image = preprocess(IMAGE_BY_NAME[f"{image_id}.jpg"])
        memory = encoder.run(["memory"], {"images": image})[0]
        beam_decode(decoder, memory)

    rows = []
    for index, image_id in enumerate(SAMPLE_IDS, start=1):
        pipeline_started = time.perf_counter()
        started = time.perf_counter()
        image = preprocess(IMAGE_BY_NAME[f"{image_id}.jpg"])
        preprocess_ms = (time.perf_counter() - started) * 1000.0
        started = time.perf_counter()
        memory = encoder.run(["memory"], {"images": image})[0]
        encoder_ms = (time.perf_counter() - started) * 1000.0
        started = time.perf_counter()
        token_ids, decoder_calls = beam_decode(decoder, memory)
        decoder_ms = (time.perf_counter() - started) * 1000.0
        end_to_end_ms = (time.perf_counter() - pipeline_started) * 1000.0
        rss_mib = (
            process.memory_info().rss / 1024**2 if process is not None else None
        )
        row = {
            "threads": thread_count,
            "image_id": image_id,
            "preprocess_ms": preprocess_ms,
            "encoder_ms": encoder_ms,
            "decoder_ms": decoder_ms,
            "end_to_end_ms": end_to_end_ms,
            "decoder_calls": decoder_calls,
            "generated_tokens": len(token_ids) - 1,
            "caption": decode(token_ids),
            "process_rss_MiB": rss_mib,
        }
        rows.append(row)
        all_rows.append(row)
        if index % 10 == 0:
            print(
                f"threads={thread_count} | {index:02d}/50 | "
                f"e2e={end_to_end_ms:.1f} ms",
                flush=True,
            )

    # Lặp lại cùng một ảnh để kiểm tra tính tất định.
    test_image = preprocess(IMAGE_BY_NAME[f"{SAMPLE_IDS[0]}.jpg"])
    test_memory = encoder.run(["memory"], {"images": test_image})[0]
    first = beam_decode(decoder, test_memory)[0]
    second = beam_decode(decoder, test_memory)[0]
    deterministic = first == second
    assert deterministic

    summaries[str(thread_count)] = {
        "session_load_ms": load_ms,
        "preprocess": percentile_summary([row["preprocess_ms"] for row in rows]),
        "encoder": percentile_summary([row["encoder_ms"] for row in rows]),
        "decoder_beam3": percentile_summary([row["decoder_ms"] for row in rows]),
        "end_to_end": percentile_summary([row["end_to_end_ms"] for row in rows]),
        "maximum_process_rss_MiB": (
            max(row["process_rss_MiB"] for row in rows)
            if process is not None else None
        ),
        "repeat_deterministic": deterministic,
    }

raw_path = WORK_ROOT / f"{EXPERIMENT_ID}_RAW.csv"
pd.DataFrame(all_rows).to_csv(raw_path, index=False, encoding="utf-8")
print("Benchmark              : COMPLETE")
print("Thread configurations  :", thread_values)

threads=1 | 10/50 | e2e=616.6 ms
threads=1 | 20/50 | e2e=931.2 ms
threads=1 | 30/50 | e2e=693.5 ms
threads=1 | 40/50 | e2e=620.8 ms
threads=1 | 50/50 | e2e=576.7 ms
threads=4 | 10/50 | e2e=425.8 ms
threads=4 | 20/50 | e2e=642.3 ms
threads=4 | 30/50 | e2e=483.6 ms
threads=4 | 40/50 | e2e=415.2 ms
threads=4 | 50/50 | e2e=384.1 ms
Benchmark              : COMPLETE
Thread configurations  : [1, 4]


## Cell 4 — Proxy report and Android handoff

In [4]:
result = {
    "schema_version": "f30k_t3s3_cpu_proxy_benchmark_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "role": "x86 Kaggle CPU proxy; not an Android device benchmark",
    "host": {
        "logical_cpu_count": os.cpu_count(),
        "onnxruntime_version": ort.__version__,
        "provider": "CPUExecutionProvider",
    },
    "sample": {
        "split": "Flickr30k Karpathy Validation",
        "images": len(SAMPLE_IDS),
        "selection": "50 evenly spaced sorted image IDs",
        "warmup_images_per_thread_configuration": 3,
    },
    "package": {
        "model_only_MiB": LOCK["size"]["selected_MiB"],
        "size_reduction_fraction": LOCK["size"]["reduction_fraction"],
        "encoder_sha256": sha256(ENCODER_PATH),
        "decoder_sha256": sha256(DECODER_PATH),
    },
    "summaries": summaries,
    "limitations": [
        "Kaggle x86 CPU timings cannot substitute for Android measurements.",
        "RSS includes Python, ONNX Runtime, and process overhead.",
        "Thermal throttling, NNAPI availability, battery and device SoC are not measured.",
    ],
    "held_out_test_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "next_phase_required": "F30K_T3S3_Android_OnDevice_Benchmark",
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json_atomic(result_path, result)
manifest_files = [raw_path, result_path]
write_json_atomic(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3s3_cpu_proxy_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": [
        {"relative_path": path.name, "size_bytes": path.stat().st_size,
         "sha256": sha256(path)}
        for path in manifest_files
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Status                 : COMPLETE")
for threads, summary in summaries.items():
    print(
        f"Threads {threads} e2e p50/p95:",
        summary["end_to_end"]["p50_ms"],
        "/",
        summary["end_to_end"]["p95_ms"],
        "ms",
    )
print("Interpretation         : CPU PROXY ONLY")
print("Held-out Test          : SEALED / NOT USED")
print("Next                   :", result["next_phase_required"])
print("Output                 :", WORK_ROOT)

Status                 : COMPLETE
Threads 1 e2e p50/p95: 652.9017420000116 / 815.5105114999969 ms
Threads 4 e2e p50/p95: 431.7086270000061 / 583.2806834500161 ms
Interpretation         : CPU PROXY ONLY
Held-out Test          : SEALED / NOT USED
Next                   : F30K_T3S3_Android_OnDevice_Benchmark
Output                 : /kaggle/working/F30K_T3S3_CPU_Proxy
